In [0]:
# ======================================================================
# CMD 04.01 — LOAD H₂ MODELING DATA
# ======================================================================

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.impute import SimpleImputer
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GroupKFold,
    cross_validate,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.svm import SVR

# XGBoost
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("WARNING: xgboost is not available.")

# ------------------------------------------------------------
# Load material + thermodynamic tables from Notebook 03
# ------------------------------------------------------------

hydrogen_train = (
    spark.table(
        "ml_perovskites.ml_hydrogen_thermo_train"
    )
    .toPandas()
)

hydrogen_test = (
    spark.table(
        "ml_perovskites.ml_hydrogen_thermo_test"
    )
    .toPandas()
)

# ------------------------------------------------------------
# Load original experimental metadata
# ------------------------------------------------------------

db_ml = (
    spark.table(
        "ml_perovskites.db_ml"
    )
    .toPandas()
)

print("=" * 70)
print("H₂ MODELING DATA")
print("=" * 70)

print("\nMaterial + thermodynamic train:")
print(hydrogen_train.shape)

print("\nMaterial + thermodynamic test:")
print(hydrogen_test.shape)

print("\nOriginal metadata:")
print(db_ml.shape)

print("\nTrain unique compounds:")
print(
    hydrogen_train["Compound"].nunique()
)

print("\nTest unique compounds:")
print(
    hydrogen_test["Compound"].nunique()
)

# ------------------------------------------------------------
# Verify train/test compound separation
# ------------------------------------------------------------

train_compounds = set(
    hydrogen_train["Compound"]
)

test_compounds = set(
    hydrogen_test["Compound"]
)

compound_overlap = (
    train_compounds
    .intersection(test_compounds)
)

print(
    "\nTrain/test compound overlap:",
    len(compound_overlap)
)

assert len(compound_overlap) == 0

print(
    "\nPASS — Material/thermodynamic train/test "
    "data have zero compound overlap."
)

In [0]:
# ======================================================================
# CMD 04.02 — DEFINE MATERIAL, THERMODYNAMIC AND EXPERIMENTAL FEATURES
# ======================================================================

TARGET_COL = "hydrogen_target"
COMPOUND_COL = "Compound"

# ------------------------------------------------------------
# Material descriptors
# ------------------------------------------------------------

material_features = sorted([
    c
    for c in hydrogen_train.columns
    if c.startswith("mean_")
])

# ------------------------------------------------------------
# Thermodynamic descriptors
# ------------------------------------------------------------

thermo_features = [
    "thermo_mean_MW",
    "thermo_mean_Tm",
    "thermo_mean_Tb",
    "thermo_mean_Hfus",
    "thermo_mean_Hvap"
]

# ------------------------------------------------------------
# Combined material block
# ------------------------------------------------------------

material_thermo_features = (
    material_features
    + thermo_features
)

# ------------------------------------------------------------
# Experimental metadata
#
# These are measured/recorded experimental conditions and are
# legitimate predictors for our experiment-level objective.
# ------------------------------------------------------------

experimental_numeric_candidates = [
    "particle_size",
    "radiation_source_intensity",
    "specific_surface_area",
    "catalyst_amount",
    "reaction_solution_concentration",
    "calcination_temperature",
    "calcination_time"
]

experimental_categorical_candidates = [
    "incident_light_cut_off",
    "co_catalyst",
    "co_catalyst_type",
    "reaction_type",
    "reaction_solution",
    "carbon_source",
    "sacrificial_agent",
    "sacrificial_agent_source",
    "preparation_method"
]

# ------------------------------------------------------------
# Keep only columns that actually exist in db_ml
# ------------------------------------------------------------

experimental_numeric_features = [
    c
    for c in experimental_numeric_candidates
    if c in db_ml.columns
]

experimental_categorical_features = [
    c
    for c in experimental_categorical_candidates
    if c in db_ml.columns
]

print("=" * 70)
print("FEATURE GROUPS")
print("=" * 70)

print(
    "\nMaterial descriptors:",
    len(material_features)
)

print(
    "Thermodynamic descriptors:",
    len(thermo_features)
)

print(
    "Material + thermodynamic:",
    len(material_thermo_features)
)

print(
    "\nExperimental numeric:",
    len(experimental_numeric_features)
)

print(
    experimental_numeric_features
)

print(
    "\nExperimental categorical:",
    len(experimental_categorical_features)
)

print(
    experimental_categorical_features
)

# ------------------------------------------------------------
# Expected material structure
# ------------------------------------------------------------

assert len(material_features) == 43
assert len(thermo_features) == 5
assert len(material_thermo_features) == 48

assert len(experimental_numeric_features) > 0
assert len(experimental_categorical_features) > 0

print(
    "\nPASS — Material/thermodynamic feature structure validated."
)

In [0]:
# ======================================================================
# CMD 04.03 — BUILD EXPERIMENT-LEVEL H₂ METADATA
# ======================================================================

print("=" * 70)
print("BUILD EXPERIMENT-LEVEL METADATA")
print("=" * 70)

# ------------------------------------------------------------
# Select only the metadata needed for modeling
# ------------------------------------------------------------

metadata_columns = (
    [COMPOUND_COL]
    + experimental_numeric_features
    + experimental_categorical_features
)

metadata = (
    db_ml[
        metadata_columns
    ]
    .copy()
)

# ------------------------------------------------------------
# Normalize compound identifiers
# ------------------------------------------------------------

metadata[COMPOUND_COL] = (
    metadata[COMPOUND_COL]
    .astype(str)
    .str.strip()
)

hydrogen_train[COMPOUND_COL] = (
    hydrogen_train[COMPOUND_COL]
    .astype(str)
    .str.strip()
)

hydrogen_test[COMPOUND_COL] = (
    hydrogen_test[COMPOUND_COL]
    .astype(str)
    .str.strip()
)

# ------------------------------------------------------------
# IMPORTANT:
#
# A compound can occur in multiple experiments.
# Therefore a compound-only merge is potentially one-to-many.
#
# We first inspect the multiplicity.
# ------------------------------------------------------------

metadata_compound_counts = (
    metadata
    .groupby(COMPOUND_COL)
    .size()
    .sort_values(ascending=False)
)

print(
    "\nMetadata rows:",
    len(metadata)
)

print(
    "Unique metadata compounds:",
    metadata[COMPOUND_COL].nunique()
)

print(
    "\nMaximum metadata rows per compound:",
    metadata_compound_counts.max()
)

print(
    "\nCompounds with >1 metadata row:",
    (
        metadata_compound_counts > 1
    ).sum()
)

# ------------------------------------------------------------
# We DO NOT merge yet.
#
# If the source metadata has multiple experimental rows per
# compound, a compound-only merge would destroy the experiment-
# level unit of analysis.
# ------------------------------------------------------------

print(
    "\nMetadata multiplicity audit complete."
)

print(
    "\nIMPORTANT:"
)

print(
    "Experimental metadata must be aligned to the exact "
    "experimental/modeling row before modeling."
)

In [0]:
# ======================================================================
# CMD 04.04 — IDENTIFY EXPERIMENT-LEVEL JOIN KEY
# ======================================================================

print("=" * 70)
print("IDENTIFY EXPERIMENT-LEVEL JOIN KEY")
print("=" * 70)

# ------------------------------------------------------------
# 1. Show all columns from the original H₂ source table
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ORIGINAL db_ml COLUMNS")
print("=" * 70)

for i, c in enumerate(db_ml.columns):
    print(f"{i:03d} | {c}")


# ------------------------------------------------------------
# 2. Show columns from the current H₂ modeling train table
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CURRENT hydrogen_train COLUMNS")
print("=" * 70)

for i, c in enumerate(hydrogen_train.columns):
    print(f"{i:03d} | {c}")


# ------------------------------------------------------------
# 3. Look specifically for identifier-like columns
# ------------------------------------------------------------

identifier_keywords = [
    "id",
    "experiment",
    "exp",
    "doi",
    "reference",
    "sample",
    "record",
    "index",
    "row"
]

db_identifier_candidates = [
    c for c in db_ml.columns
    if any(
        keyword in c.lower()
        for keyword in identifier_keywords
    )
]

train_identifier_candidates = [
    c for c in hydrogen_train.columns
    if any(
        keyword in c.lower()
        for keyword in identifier_keywords
    )
]

print("\n" + "=" * 70)
print("IDENTIFIER-LIKE COLUMNS — db_ml")
print("=" * 70)

print(db_identifier_candidates)

print("\n" + "=" * 70)
print("IDENTIFIER-LIKE COLUMNS — hydrogen_train")
print("=" * 70)

print(train_identifier_candidates)


# ------------------------------------------------------------
# 4. Inspect first rows of candidate columns
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("db_ml IDENTIFIER CANDIDATE PREVIEW")
print("=" * 70)

if db_identifier_candidates:
    display(
        db_ml[
            db_identifier_candidates
        ].head(20)
    )
else:
    print("No identifier-like columns found.")


print("\n" + "=" * 70)
print("hydrogen_train IDENTIFIER CANDIDATE PREVIEW")
print("=" * 70)

if train_identifier_candidates:
    display(
        hydrogen_train[
            train_identifier_candidates
        ].head(20)
    )
else:
    print("No identifier-like columns found.")

In [0]:
# ======================================================================
# CMD 04.05 — VERIFY COMPOUND + DOI EXPERIMENT KEY
# ======================================================================

print("=" * 70)
print("VERIFY COMPOUND + DOI EXPERIMENT KEY")
print("=" * 70)

KEY_COLS = [
    "Compound",
    "doi_reference"
]

# ------------------------------------------------------------
# Normalize key columns
# ------------------------------------------------------------

db_ml_keycheck = db_ml.copy()

for c in KEY_COLS:
    db_ml_keycheck[c] = (
        db_ml_keycheck[c]
        .astype("string")
        .str.strip()
    )

# ------------------------------------------------------------
# Number of unique combinations
# ------------------------------------------------------------

n_rows = len(db_ml_keycheck)

n_unique_compound = (
    db_ml_keycheck["Compound"]
    .nunique(dropna=False)
)

n_unique_doi = (
    db_ml_keycheck["doi_reference"]
    .nunique(dropna=False)
)

n_unique_pair = (
    db_ml_keycheck[KEY_COLS]
    .drop_duplicates()
    .shape[0]
)

print("\nRows in db_ml:", n_rows)
print("Unique compounds:", n_unique_compound)
print("Unique DOIs:", n_unique_doi)
print("Unique Compound + DOI pairs:", n_unique_pair)

# ------------------------------------------------------------
# Check duplicated Compound + DOI combinations
# ------------------------------------------------------------

pair_counts = (
    db_ml_keycheck
    .groupby(
        KEY_COLS,
        dropna=False
    )
    .size()
    .reset_index(name="n_rows")
)

duplicated_pairs = (
    pair_counts[
        pair_counts["n_rows"] > 1
    ]
    .sort_values(
        "n_rows",
        ascending=False
    )
)

print("\n" + "=" * 70)
print("DUPLICATED COMPOUND + DOI PAIRS")
print("=" * 70)

print(
    "Number of duplicated pairs:",
    len(duplicated_pairs)
)

print(
    "Rows represented by duplicated pairs:",
    duplicated_pairs["n_rows"].sum()
    if len(duplicated_pairs) > 0
    else 0
)

if len(duplicated_pairs) > 0:
    display(
        duplicated_pairs.head(30)
    )

# ------------------------------------------------------------
# Inspect whether duplicated pairs actually have different
# experimental conditions / H₂ measurements
# ------------------------------------------------------------

experimental_check_cols = [
    "hydrogen_production_rate",
    "incident_light_cut_off",
    "particle_size",
    "radiation_source_intensity",
    "specific_surface_area",
    "catalyst_amount",
    "co_catalyst",
    "co_catalyst_type",
    "reaction_type",
    "reaction_solution",
    "reaction_solution_concentration",
    "carbon_source",
    "sacrificial_agent",
    "sacrificial_agent_source",
    "preparation_method",
    "calcination_temperature",
    "calcination_time"
]

experimental_check_cols = [
    c
    for c in experimental_check_cols
    if c in db_ml_keycheck.columns
]

if len(duplicated_pairs) > 0:

    duplicate_pair_keys = duplicated_pairs[
        KEY_COLS
    ]

    duplicated_records = (
        db_ml_keycheck
        .merge(
            duplicate_pair_keys,
            on=KEY_COLS,
            how="inner"
        )
        .sort_values(
            KEY_COLS
        )
    )

    print("\n" + "=" * 70)
    print("EXPERIMENTAL RECORDS FOR DUPLICATED PAIRS")
    print("=" * 70)

    display(
        duplicated_records[
            KEY_COLS + experimental_check_cols
        ].head(100)
    )

# ------------------------------------------------------------
# Check whether Compound + DOI uniquely identifies the rows
# ------------------------------------------------------------

if n_unique_pair == n_rows:

    print("\n" + "=" * 70)
    print("RESULT")
    print("=" * 70)

    print(
        "PASS — Compound + DOI uniquely identifies "
        "every db_ml row."
    )

else:

    print("\n" + "=" * 70)
    print("RESULT")
    print("=" * 70)

    print(
        "Compound + DOI is NOT unique."
    )

    print(
        "We must resolve the duplicated pairs before "
        "merging experimental metadata."
    )

In [0]:
# ======================================================================
# CMD 04.06 — VERIFY TARGET-LEVEL EXPERIMENT ALIGNMENT
# ======================================================================

print("=" * 70)
print("VERIFY TARGET-LEVEL EXPERIMENT ALIGNMENT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Reload the original source table explicitly
# ------------------------------------------------------------

db_ml_alignment = (
    spark.table("ml_perovskites.db_ml")
    .toPandas()
    .copy()
)

# ------------------------------------------------------------
# 2. Confirm the actual source target column
# ------------------------------------------------------------

SOURCE_TARGET_COL = "hydrogen_production_rate"

if SOURCE_TARGET_COL not in db_ml_alignment.columns:
    raise RuntimeError(
        f"Expected source target column '{SOURCE_TARGET_COL}' "
        f"was not found in db_ml."
    )

# ------------------------------------------------------------
# 3. Normalize identifiers and target
# ------------------------------------------------------------

for df in [
    db_ml_alignment,
    hydrogen_train,
    hydrogen_test
]:

    df["Compound"] = (
        df["Compound"]
        .astype("string")
        .str.strip()
    )

    df["doi_reference"] = (
        df["doi_reference"]
        .astype("string")
        .str.strip()
    )

# Numeric conversion
db_ml_alignment[SOURCE_TARGET_COL] = pd.to_numeric(
    db_ml_alignment[SOURCE_TARGET_COL],
    errors="coerce"
)

hydrogen_train["hydrogen_target"] = pd.to_numeric(
    hydrogen_train["hydrogen_target"],
    errors="coerce"
)

hydrogen_test["hydrogen_target"] = pd.to_numeric(
    hydrogen_test["hydrogen_target"],
    errors="coerce"
)

# ------------------------------------------------------------
# 4. Build source-level alignment key
# ------------------------------------------------------------

db_ml_alignment["_alignment_key"] = list(
    zip(
        db_ml_alignment["Compound"],
        db_ml_alignment["doi_reference"],
        db_ml_alignment[SOURCE_TARGET_COL]
    )
)

hydrogen_train["_alignment_key"] = list(
    zip(
        hydrogen_train["Compound"],
        hydrogen_train["doi_reference"],
        hydrogen_train["hydrogen_target"]
    )
)

hydrogen_test["_alignment_key"] = list(
    zip(
        hydrogen_test["Compound"],
        hydrogen_test["doi_reference"],
        hydrogen_test["hydrogen_target"]
    )
)

# ------------------------------------------------------------
# 5. Source-key uniqueness
# ------------------------------------------------------------

source_key_counts = (
    db_ml_alignment[
        [
            "Compound",
            "doi_reference",
            SOURCE_TARGET_COL
        ]
    ]
    .value_counts(
        dropna=False
    )
    .reset_index(
        name="n_source_rows"
    )
)

duplicated_source_keys = (
    source_key_counts[
        source_key_counts["n_source_rows"] > 1
    ]
)

print("\n" + "=" * 70)
print("SOURCE KEY UNIQUENESS")
print("=" * 70)

print(
    "Source rows:",
    len(db_ml_alignment)
)

print(
    "Unique Compound + DOI + H₂ keys:",
    len(source_key_counts)
)

print(
    "Duplicated Compound + DOI + H₂ keys:",
    len(duplicated_source_keys)
)

if len(duplicated_source_keys) > 0:

    print(
        "\nDuplicated source keys:"
    )

    display(
        duplicated_source_keys.head(30)
    )

# ------------------------------------------------------------
# 6. Check whether every modeling row exists in source
# ------------------------------------------------------------

source_keys = set(
    db_ml_alignment["_alignment_key"]
)

train_keys = set(
    hydrogen_train["_alignment_key"]
)

test_keys = set(
    hydrogen_test["_alignment_key"]
)

train_missing = (
    train_keys
    - source_keys
)

test_missing = (
    test_keys
    - source_keys
)

print("\n" + "=" * 70)
print("TARGET-LEVEL ALIGNMENT")
print("=" * 70)

print(
    "Train rows:",
    len(hydrogen_train)
)

print(
    "Train unique alignment keys:",
    len(train_keys)
)

print(
    "Train keys found in source:",
    len(train_keys & source_keys)
)

print(
    "Train keys missing from source:",
    len(train_missing)
)

print(
    "\nTest rows:",
    len(hydrogen_test)
)

print(
    "Test unique alignment keys:",
    len(test_keys)
)

print(
    "Test keys found in source:",
    len(test_keys & source_keys)
)

print(
    "Test keys missing from source:",
    len(test_missing)
)

# ------------------------------------------------------------
# 7. Check duplicate keys inside modeling sets
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MODELING-SET KEY UNIQUENESS")
print("=" * 70)

for name, df in [
    ("TRAIN", hydrogen_train),
    ("TEST", hydrogen_test)
]:

    key_counts = (
        df[
            [
                "Compound",
                "doi_reference",
                "hydrogen_target"
            ]
        ]
        .value_counts(
            dropna=False
        )
        .reset_index(
            name="n_rows"
        )
    )

    duplicated = key_counts[
        key_counts["n_rows"] > 1
    ]

    print(f"\n{name}")

    print(
        "Rows:",
        len(df)
    )

    print(
        "Unique keys:",
        len(key_counts)
    )

    print(
        "Duplicated keys:",
        len(duplicated)
    )

# ------------------------------------------------------------
# 8. Show missing alignment keys if present
# ------------------------------------------------------------

if train_missing:

    print("\n" + "=" * 70)
    print("MISSING TRAIN ALIGNMENT KEYS")
    print("=" * 70)

    display(
        hydrogen_train[
            hydrogen_train["_alignment_key"].isin(
                train_missing
            )
        ][
            [
                "Compound",
                "doi_reference",
                "hydrogen_target"
            ]
        ]
    )

if test_missing:

    print("\n" + "=" * 70)
    print("MISSING TEST ALIGNMENT KEYS")
    print("=" * 70)

    display(
        hydrogen_test[
            hydrogen_test["_alignment_key"].isin(
                test_missing
            )
        ][
            [
                "Compound",
                "doi_reference",
                "hydrogen_target"
            ]
        ]
    )

# ------------------------------------------------------------
# 9. Final diagnostic
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALIGNMENT DIAGNOSTIC")
print("=" * 70)

if (
    len(duplicated_source_keys) == 0
    and len(train_missing) == 0
    and len(test_missing) == 0
):

    print(
        "PASS — Compound + DOI + H₂ target provides "
        "a unique and complete source alignment key."
    )

else:

    print(
        "FURTHER INVESTIGATION REQUIRED."
    )

# ------------------------------------------------------------
# IMPORTANT:
# Do NOT merge experimental metadata yet.
# This CMD only establishes whether the alignment key is safe.
# ------------------------------------------------------------

In [0]:
# ======================================================================
# CMD 04.07 — RESOLVE ALIGNMENT ANOMALIES
# ======================================================================

print("=" * 70)
print("RESOLVE ALIGNMENT ANOMALIES")
print("=" * 70)

# ------------------------------------------------------------
# Rebuild clean source and modeling alignment tables
# ------------------------------------------------------------

source = db_ml_alignment.copy()

train = hydrogen_train[
    [
        "Compound",
        "doi_reference",
        "hydrogen_target"
    ]
].copy()

test = hydrogen_test[
    [
        "Compound",
        "doi_reference",
        "hydrogen_target"
    ]
].copy()

# ------------------------------------------------------------
# Normalize identifiers
# ------------------------------------------------------------

for df in [source, train, test]:

    df["Compound"] = (
        df["Compound"]
        .astype("string")
        .str.strip()
    )

    df["doi_reference"] = (
        df["doi_reference"]
        .astype("string")
        .str.strip()
    )

source[SOURCE_TARGET_COL] = pd.to_numeric(
    source[SOURCE_TARGET_COL],
    errors="coerce"
)

train["hydrogen_target"] = pd.to_numeric(
    train["hydrogen_target"],
    errors="coerce"
)

test["hydrogen_target"] = pd.to_numeric(
    test["hydrogen_target"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. SOURCE DUPLICATES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. DUPLICATED SOURCE KEYS")
print("=" * 70)

source_key_cols = [
    "Compound",
    "doi_reference",
    SOURCE_TARGET_COL
]

source_counts = (
    source
    .groupby(
        source_key_cols,
        dropna=False
    )
    .size()
    .reset_index(name="n_rows")
)

duplicated_source_keys = source_counts[
    source_counts["n_rows"] > 1
].copy()

print(
    "Duplicated source keys:",
    len(duplicated_source_keys)
)

if len(duplicated_source_keys) > 0:

    display(
        duplicated_source_keys
    )

    duplicated_source_records = (
        source
        .merge(
            duplicated_source_keys[
                source_key_cols
            ],
            on=source_key_cols,
            how="inner"
        )
        .sort_values(
            source_key_cols
        )
    )

    print(
        "\nFull records belonging to duplicated keys:"
    )

    display(
        duplicated_source_records[
            [
                "Compound",
                "doi_reference",
                SOURCE_TARGET_COL,
                "incident_light_cut_off",
                "particle_size",
                "radiation_source_intensity",
                "specific_surface_area",
                "catalyst_amount",
                "co_catalyst",
                "co_catalyst_type",
                "reaction_type",
                "reaction_solution",
                "reaction_solution_concentration",
                "carbon_source",
                "sacrificial_agent",
                "sacrificial_agent_source",
                "preparation_method",
                "calcination_temperature",
                "calcination_time"
            ]
        ]
    )

# ------------------------------------------------------------
# 2. MODELING-SET DUPLICATES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. DUPLICATED MODELING KEYS")
print("=" * 70)

model_key_cols = [
    "Compound",
    "doi_reference",
    "hydrogen_target"
]

for name, df in [
    ("TRAIN", train),
    ("TEST", test)
]:

    counts = (
        df
        .groupby(
            model_key_cols,
            dropna=False
        )
        .size()
        .reset_index(name="n_rows")
    )

    duplicates = counts[
        counts["n_rows"] > 1
    ]

    print(f"\n{name}")
    print(
        "Duplicated keys:",
        len(duplicates)
    )

    if len(duplicates) > 0:

        display(
            duplicates
        )

        duplicated_rows = (
            df
            .merge(
                duplicates[
                    model_key_cols
                ],
                on=model_key_cols,
                how="inner"
            )
            .sort_values(
                model_key_cols
            )
        )

        display(
            duplicated_rows
        )

# ------------------------------------------------------------
# 3. ROBUST SOURCE-TO-MODELING MATCH
#
# Do NOT use tuple/set membership here.
# Use pandas merge with an explicit marker.
# ------------------------------------------------------------

source_match = source[
    [
        "Compound",
        "doi_reference",
        SOURCE_TARGET_COL
    ]
].copy()

source_match = (
    source_match
    .drop_duplicates()
)

train_match = (
    train
    .rename(
        columns={
            "hydrogen_target":
                SOURCE_TARGET_COL
        }
    )
)

test_match = (
    test
    .rename(
        columns={
            "hydrogen_target":
                SOURCE_TARGET_COL
        }
    )
)

# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

train_check = train_match.merge(
    source_match,
    on=[
        "Compound",
        "doi_reference",
        SOURCE_TARGET_COL
    ],
    how="left",
    indicator=True
)

train_missing = train_check[
    train_check["_merge"] == "left_only"
].copy()

print("\n" + "=" * 70)
print("3. UNMATCHED TRAIN ROWS")
print("=" * 70)

print(
    "Unmatched train rows:",
    len(train_missing)
)

if len(train_missing) > 0:

    display(
        train_missing[
            [
                "Compound",
                "doi_reference",
                SOURCE_TARGET_COL
            ]
        ]
    )

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

test_check = test_match.merge(
    source_match,
    on=[
        "Compound",
        "doi_reference",
        SOURCE_TARGET_COL
    ],
    how="left",
    indicator=True
)

test_missing = test_check[
    test_check["_merge"] == "left_only"
].copy()

print("\n" + "=" * 70)
print("4. UNMATCHED TEST ROWS")
print("=" * 70)

print(
    "Unmatched test rows:",
    len(test_missing)
)

if len(test_missing) > 0:

    display(
        test_missing[
            [
                "Compound",
                "doi_reference",
                SOURCE_TARGET_COL
            ]
        ]
    )

# ------------------------------------------------------------
# 5. SEARCH SOURCE BY COMPOUND + TARGET FOR UNMATCHED ROWS
#
# This helps distinguish:
#   - DOI mismatch
#   - duplicated observations
#   - target representation issue
# ------------------------------------------------------------

def inspect_unmatched(unmatched, label):

    if len(unmatched) == 0:
        return

    print("\n" + "=" * 70)
    print(
        f"5. SOURCE SEARCH FOR UNMATCHED {label} ROWS"
    )
    print("=" * 70)

    for _, row in unmatched.iterrows():

        compound = row["Compound"]
        target = row[SOURCE_TARGET_COL]

        candidates = source[
            (
                source["Compound"]
                == compound
            )
            &
            (
                np.isclose(
                    source[SOURCE_TARGET_COL],
                    target,
                    equal_nan=True
                )
            )
        ].copy()

        print(
            f"\nCompound: {compound}"
        )

        print(
            f"Target: {target}"
        )

        if len(candidates) == 0:

            print(
                "No source row with matching "
                "Compound + H₂ target."
            )

        else:

            display(
                candidates[
                    [
                        "Compound",
                        "doi_reference",
                        SOURCE_TARGET_COL,
                        "incident_light_cut_off",
                        "particle_size",
                        "radiation_source_intensity",
                        "specific_surface_area",
                        "catalyst_amount",
                        "co_catalyst",
                        "co_catalyst_type",
                        "reaction_type",
                        "reaction_solution",
                        "reaction_solution_concentration",
                        "carbon_source",
                        "sacrificial_agent",
                        "sacrificial_agent_source",
                        "preparation_method",
                        "calcination_temperature",
                        "calcination_time"
                    ]
                ]
            )


inspect_unmatched(
    train_missing,
    "TRAIN"
)

inspect_unmatched(
    test_missing,
    "TEST"
)

print("\n" + "=" * 70)
print("ALIGNMENT ANOMALY INSPECTION COMPLETE")
print("=" * 70)

In [0]:
# ======================================================================
# CMD 04.08 — DEFINE MODELING TARGETS
# ======================================================================

y_train_original = (
    hydrogen_train[
        target_column
    ]
    .astype(float)
    .values
)

y_test_original = (
    hydrogen_test[
        target_column
    ]
    .astype(float)
    .values
)

y_train_log = np.log1p(
    y_train_original
)

y_test_log = np.log1p(
    y_test_original
)

print("=" * 70)
print("H₂ TARGETS")
print("=" * 70)

print(
    "\nOriginal target:"
)

print(
    "Train min:",
    y_train_original.min()
)

print(
    "Train max:",
    y_train_original.max()
)

print(
    "Test min:",
    y_test_original.min()
)

print(
    "Test max:",
    y_test_original.max()
)

print(
    "\nLog-transformed target:"
)

print(
    "Train min:",
    y_train_log.min()
)

print(
    "Train max:",
    y_train_log.max()
)

print(
    "Test min:",
    y_test_log.min()
)

print(
    "Test max:",
    y_test_log.max()
)

print(
    "\nZero targets — train:",
    (y_train_original == 0).sum()
)

print(
    "Zero targets — test:",
    (y_test_original == 0).sum()
)

In [0]:
# ======================================================================
# CMD 04.09 — CROSS-VALIDATE MI FEATURE SUBSETS WITHOUT LEAKAGE
# ======================================================================

from sklearn.feature_selection import SelectKBest

# ------------------------------------------------------------
# Feature-selection function
# ------------------------------------------------------------

def mutual_information_score(
    X,
    y
):
    return mutual_info_regression(
        X,
        y,
        random_state=42
    )

# ------------------------------------------------------------
# Candidate feature counts
# ------------------------------------------------------------

MI_TOP_K = [
    10,
    15,
    20,
    25,
    30,
    35,
    41,
    48
]

# ------------------------------------------------------------
# Evaluate MI-selected subsets
#
# Feature selection happens INSIDE each CV fold.
# Therefore the validation fold never influences
# which features are selected.
# ------------------------------------------------------------

mi_cv_results = []

for k in MI_TOP_K:

    selector = SelectKBest(
        score_func=mutual_information_score,
        k=k
    )

    model = Pipeline([
        (
            "feature_selection",
            selector
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KernelRidge(
                alpha=1.0,
                kernel="rbf"
            )
        )
    ])

    scores = cross_validate(
        model,
        X_train_candidates,
        y_train_log,
        groups=groups_train,
        cv=group_cv,
        scoring="r2",
        n_jobs=-1,
        return_train_score=False
    )

    mi_cv_results.append({
        "n_features": k,
        "mean_cv_r2": scores[
            "test_score"
        ].mean(),
        "std_cv_r2": scores[
            "test_score"
        ].std()
    })

mi_cv_results = (
    pd.DataFrame(mi_cv_results)
    .sort_values(
        "mean_cv_r2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("MI FEATURE SUBSET CV")
print("=" * 70)

display(
    mi_cv_results
)

In [0]:
# ======================================================================
# CMD 04.10 — COMPARE FEATURE SUBSETS ACROSS MODELS
# ======================================================================

models = {

    "KRR": Pipeline([
        (
            "feature_selection",
            SelectKBest(
                score_func=mutual_information_score,
                k=10
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KernelRidge(
                alpha=1.0,
                kernel="rbf"
            )
        )
    ]),

    "SVR": Pipeline([
        (
            "feature_selection",
            SelectKBest(
                score_func=mutual_information_score,
                k=10
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            SVR(
                kernel="rbf",
                C=10.0,
                epsilon=0.1
            )
        )
    ]),

    "RandomForest": Pipeline([
        (
            "feature_selection",
            SelectKBest(
                score_func=mutual_information_score,
                k=10
            )
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]),

    "GradientBoosting": Pipeline([
        (
            "feature_selection",
            SelectKBest(
                score_func=mutual_information_score,
                k=10
            )
        ),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=300,
                learning_rate=0.03,
                max_depth=3,
                random_state=42
            )
        )
    ])
}

multi_model_results = []

for k in MI_TOP_K:

    for model_name, base_model in models.items():

        model = clone(
            base_model
        )

        model.set_params(
            feature_selection__k=k
        )

        scores = cross_validate(
            model,
            X_train_candidates,
            y_train_log,
            groups=groups_train,
            cv=group_cv,
            scoring="r2",
            n_jobs=-1,
            return_train_score=False
        )

        multi_model_results.append({
            "model": model_name,
            "n_features": k,
            "mean_cv_r2": scores[
                "test_score"
            ].mean(),
            "std_cv_r2": scores[
                "test_score"
            ].std()
        })

multi_model_results = (
    pd.DataFrame(
        multi_model_results
    )
    .sort_values(
        "mean_cv_r2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("GROUPED CV — MODEL / FEATURE COMPARISON")
print("=" * 70)

display(
    multi_model_results
)

In [0]:
# ======================================================================
# CMD 04.11 — OPTIMIZE RANDOM FOREST WITH GROUPED CV
# ======================================================================

from sklearn.model_selection import RandomizedSearchCV

# ------------------------------------------------------------
# Identify best RF feature count from grouped CV
# ------------------------------------------------------------

rf_results = multi_model_results[
    multi_model_results["model"]
    == "RandomForest"
].copy()

best_rf_feature_count = int(
    rf_results.iloc[0]["n_features"]
)

print(
    "Best RF feature count from grouped CV:",
    best_rf_feature_count
)

# ------------------------------------------------------------
# RF estimator
# ------------------------------------------------------------

rf_model = Pipeline([
    (
        "feature_selection",
        SelectKBest(
            score_func=mutual_information_score,
            k=best_rf_feature_count
        )
    ),
    (
        "model",
        RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        )
    )
])

# ------------------------------------------------------------
# Parameter search
# ------------------------------------------------------------

rf_param_grid = {
    "model__n_estimators": [
        300,
        500,
        800
    ],
    "model__max_depth": [
        None,
        5,
        10,
        15
    ],
    "model__min_samples_leaf": [
        1,
        2,
        5
    ],
    "model__max_features": [
        "sqrt",
        0.7,
        1.0
    ]
}

rf_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_param_grid,
    n_iter=20,
    cv=group_cv,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    refit=True,
    return_train_score=False
)

rf_search.fit(
    X_train_candidates,
    y_train_log,
    groups=groups_train
)

print(
    "\nBest grouped-CV R²:",
    rf_search.best_score_
)

print(
    "\nBest parameters:"
)

print(
    rf_search.best_params_
)

In [0]:
# ======================================================================
# CMD 04.12 — FINALIZE RANDOM FOREST ON FULL TRAINING DATA
# ======================================================================

# ------------------------------------------------------------
# Best model from grouped CV
# ------------------------------------------------------------

final_rf = rf_search.best_estimator_

# ------------------------------------------------------------
# Fit on ALL training compounds
#
# The test set remains untouched.
# ------------------------------------------------------------

final_rf.fit(
    X_train_candidates,
    y_train_log
)

print("=" * 70)
print("FINAL H₂ RANDOM FOREST")
print("=" * 70)

print(
    "\nBest grouped-CV R²:",
    rf_search.best_score_
)

print(
    "\nBest parameters:"
)

print(
    rf_search.best_params_
)

print(
    "\nFinal model fitted on:",
    len(hydrogen_train),
    "training rows"
)

print(
    "Training compounds:",
    hydrogen_train["Compound"].nunique()
)

print(
    "\nPASS — Final RF fitted without using the test set."
)

In [0]:
# ======================================================================
# CMD 04.13 — FINAL TEST EVALUATION
# ======================================================================

# ------------------------------------------------------------
# Predict untouched test set
# ------------------------------------------------------------

y_pred_log = final_rf.predict(
    X_test_candidates
)

y_pred_original = np.expm1(
    y_pred_log
)

# ------------------------------------------------------------
# Prevent numerical negative predictions
# ------------------------------------------------------------

y_pred_original = np.maximum(
    y_pred_original,
    0
)

# ------------------------------------------------------------
# Metrics — original H₂ scale
# ------------------------------------------------------------

test_r2_original = r2_score(
    y_test_original,
    y_pred_original
)

test_mae_original = mean_absolute_error(
    y_test_original,
    y_pred_original
)

test_rmse_original = np.sqrt(
    mean_squared_error(
        y_test_original,
        y_pred_original
    )
)

# ------------------------------------------------------------
# Metrics — log scale
# ------------------------------------------------------------

test_r2_log = r2_score(
    y_test_log,
    y_pred_log
)

test_mae_log = mean_absolute_error(
    y_test_log,
    y_pred_log
)

test_rmse_log = np.sqrt(
    mean_squared_error(
        y_test_log,
        y_pred_log
    )
)

print("=" * 70)
print("FINAL H₂ TEST EVALUATION")
print("=" * 70)

print("\nOriginal H₂ scale")
print("-----------------")

print(
    f"R²   : {test_r2_original:.6f}"
)

print(
    f"MAE  : {test_mae_original:.6f}"
)

print(
    f"RMSE : {test_rmse_original:.6f}"
)

print("\nLog1p(H₂) scale")
print("----------------")

print(
    f"R²   : {test_r2_log:.6f}"
)

print(
    f"MAE  : {test_mae_log:.6f}"
)

print(
    f"RMSE : {test_rmse_log:.6f}"
)

In [0]:
# ======================================================================
# CMD 04.14 — TEST PREDICTION AUDIT
# ======================================================================

prediction_audit = pd.DataFrame({
    "compound": hydrogen_test["Compound"].values,
    "actual_h2": y_test_original,
    "predicted_h2": y_pred_original
})

prediction_audit["absolute_error"] = (
    prediction_audit["actual_h2"]
    -
    prediction_audit["predicted_h2"]
).abs()

prediction_audit["log_actual"] = (
    np.log1p(
        prediction_audit["actual_h2"]
    )
)

prediction_audit["log_predicted"] = (
    np.log1p(
        prediction_audit["predicted_h2"]
    )
)

prediction_audit["log_absolute_error"] = (
    prediction_audit["log_actual"]
    -
    prediction_audit["log_predicted"]
).abs()

print("=" * 70)
print("TEST PREDICTION AUDIT")
print("=" * 70)

print("\nLargest absolute errors:")

display(
    prediction_audit
    .sort_values(
        "absolute_error",
        ascending=False
    )
    .head(15)
)

print("\nTest target statistics:")

display(
    prediction_audit[
        "actual_h2"
    ].describe()
)

print("\nPrediction statistics:")

display(
    prediction_audit[
        "predicted_h2"
    ].describe()
)

print(
    "\nTest compounds:",
    prediction_audit["compound"].nunique()
)

print(
    "\nPASS — Prediction audit complete."
)

In [0]:
# ======================================================================
# CMD 04.15 — MATERIAL-ONLY VS THERMODYNAMIC-ENRICHED CV
# ======================================================================

# ------------------------------------------------------------
# Compare two scientifically meaningful feature groups:
#
# 1. Material descriptors only
# 2. Material + thermodynamic descriptors
# ------------------------------------------------------------

feature_group_definitions = {
    "Material_only_43": material_features,
    "Material_plus_Thermo_48": combined_features
}

thermo_comparison_results = []

for group_name, features in feature_group_definitions.items():

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            KernelRidge(
                alpha=1.0,
                kernel="rbf"
            )
        )
    ])

    scores = cross_validate(
        model,
        hydrogen_train[features],
        y_train_log,
        groups=groups_train,
        cv=group_cv,
        scoring="r2",
        n_jobs=-1,
        return_train_score=False
    )

    thermo_comparison_results.append({
        "feature_group": group_name,
        "n_features": len(features),
        "mean_cv_r2": scores[
            "test_score"
        ].mean(),
        "std_cv_r2": scores[
            "test_score"
        ].std()
    })

thermo_comparison_results = (
    pd.DataFrame(
        thermo_comparison_results
    )
)

print("=" * 70)
print("THERMODYNAMIC FEATURE VALUE — GROUPED CV")
print("=" * 70)

display(
    thermo_comparison_results
)

In [0]:
# ======================================================================
# CMD 04.16 — MATERIAL-ONLY VS THERMODYNAMIC-ENRICHED RANDOM FOREST
# ======================================================================

rf_thermo_comparison = []

for group_name, features in feature_group_definitions.items():

    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    )

    scores = cross_validate(
        model,
        hydrogen_train[features],
        y_train_log,
        groups=groups_train,
        cv=group_cv,
        scoring="r2",
        n_jobs=-1,
        return_train_score=False
    )

    rf_thermo_comparison.append({
        "feature_group": group_name,
        "n_features": len(features),
        "mean_cv_r2": scores[
            "test_score"
        ].mean(),
        "std_cv_r2": scores[
            "test_score"
        ].std()
    })

rf_thermo_comparison = (
    pd.DataFrame(
        rf_thermo_comparison
    )
)

print("=" * 70)
print("THERMODYNAMIC FEATURE VALUE — RANDOM FOREST")
print("=" * 70)

display(
    rf_thermo_comparison
)

In [0]:
# ======================================================================
# CMD 04.19 — COMPARE TRAIN/TEST H₂ DISTRIBUTION
# ======================================================================

distribution_comparison = pd.DataFrame({
    "statistic": [
        "count",
        "zero_count",
        "minimum",
        "25th_percentile",
        "median",
        "75th_percentile",
        "90th_percentile",
        "95th_percentile",
        "99th_percentile",
        "maximum"
    ],

    "train": [
        len(hydrogen_train),

        (
            hydrogen_train[target_column] == 0
        ).sum(),

        hydrogen_train[target_column].min(),

        hydrogen_train[target_column].quantile(
            0.25
        ),

        hydrogen_train[target_column].median(),

        hydrogen_train[target_column].quantile(
            0.75
        ),

        hydrogen_train[target_column].quantile(
            0.90
        ),

        hydrogen_train[target_column].quantile(
            0.95
        ),

        hydrogen_train[target_column].quantile(
            0.99
        ),

        hydrogen_train[target_column].max()
    ],

    "test": [
        len(hydrogen_test),

        (
            hydrogen_test[target_column] == 0
        ).sum(),

        hydrogen_test[target_column].min(),

        hydrogen_test[target_column].quantile(
            0.25
        ),

        hydrogen_test[target_column].median(),

        hydrogen_test[target_column].quantile(
            0.75
        ),

        hydrogen_test[target_column].quantile(
            0.90
        ),

        hydrogen_test[target_column].quantile(
            0.95
        ),

        hydrogen_test[target_column].quantile(
            0.99
        ),

        hydrogen_test[target_column].max()
    ]
})

print("=" * 70)
print("TRAIN / TEST H₂ DISTRIBUTION")
print("=" * 70)

display(
    distribution_comparison
)